# Deep Learning on Meshes and Point Clouds
---
## Tutorial Part 2

In part 2 of this tutorial, we learn more about DiffusionNet, a network built with geometric operators such as Laplacians and Gradients. We will inspect the types of features that can be learned and understand how a forward pass is implemented for such operations.

During the tutorial, you will find some lines marked with $\rightarrow$.<br />
$\rightarrow$ Pause and try out the tasks to get a deeper understanding.

You should already have a working environment from part 1. Import the required modules to see if your environment is set up correctly. The DiffusionNet code should be in your repository as a git submodule. If you see that the `diffusion_net` module is not loading, you can pull the submodules with
```
git submodule update --recursive
```

In [ ]:
import torch
import torch_geometric as pyg
import polyscope as ps
import meshplot as mp
from tqdm.notebook import tqdm

import sys, os
sys.path.append(os.path.join("diffusion-net/src"))  # add the path to the DiffusionNet src
import diffusion_net

### Testing the environment
We want to make sure that all the libraries we installed work (we expect no errors when running the imports) and the CUDA components are available, if you have a GPU.

Do not worry if CUDA support says `False`, you can still run the tutorial on your CPU.

In [ ]:
print(f"CUDA support: {torch.cuda.is_available()}")

## Step 1: Load a dataset.

We will use the small ModelNet10 dataset again from Part 1, so no need to download again. We can simply load the dataset.

In [ ]:
import torch_geometric.transforms as T

# Convert the input to a point cloud
pre_transform = T.Compose((
    T.NormalizeScale(),
    T.SamplePoints(1024), # If we want to sample more points, we can change this number
))
# Add random augmentation during training
transform = T.Compose((
    T.RandomScale((0.85, 1.15)),
    T.RandomRotate(45, axis=0),
    T.RandomRotate(45, axis=1),
    T.RandomRotate(45, axis=2))
)
# We load the dataset with the pre-transforms and the transform
train_dataset = pyg.datasets.ModelNet(root='data', name='10', train=True, transform=transform, pre_transform=pre_transform, force_reload=True)
# And also the test dataset. Note that we do not apply the data augmentation transforms to the test set
test_dataset = pyg.datasets.ModelNet(root='data', name='10', train=False, transform=transform, pre_transform=pre_transform)

As a sanity check, we plot the first model in the dataset.

In [ ]:
index = 0
ps.init()
ps.register_point_cloud("ModelNet Object", 
                         # We need to convert from torch to numpy for polyscope
                          train_dataset[index].pos.numpy())
ps.show()

If you're working in the cloud on a Jupter notebook, it's easier to use `meshplot`, because it runs directly within Jupyter. We'll continue with `meshplot` in the remainder, so you can also follow along in a cloud hosted notebook.

$\rightarrow$ Try plotting a couple of different meshes.

In [ ]:
index = 0
mp.plot(train_dataset[index].pos.numpy(), shading={"point_size": 0.1})

## Step 2: Understanding DiffusionNet
DiffusionNet is a method for deep learning on surfaces that is designed to be agnostic to the data representation. You can learn more about the method [in the paper](https://arxiv.org/abs/2012.00888) or [the author's GitHub repository](https://github.com/nmwsharp/diffusion-net). We will first inspect the core operations of DiffusionNet and then set up a model with DiffusionNet.

DiffusionNet creates convolution-like blocks with geometric operators: the Laplacian and gradient. Let's compute these operators and see what they do.

In [ ]:
from diffusion_net.geometry import get_operators

index = 0
verts = train_dataset[index].pos
faces = torch.tensor([], dtype=torch.long)
# Construct the geometric operators used for DiffusionNet
frames, mass, L, evals, evecs, gradX, gradY = \
        get_operators(verts, faces)

The output of this function are the following:
- `frames` (Nx3x3 tensor) tangent frame (3x3) per vertex (useful for visualizing gradient features);
- `mass` (N vector) the mass for each vertex, corresponds to the area covered by a vertex (1/3 of the adjacent triangle areas);
- `L` (NxN matrix) the cotan Laplacian (a NxN matrix);
- `evals` (128 vector) the lowest 128 eigenvalues of the Laplacian (used to compute diffusion);
- `evecs` (Nx128 matrix) the eigenvectors corresponding to the lowest 128 eigenvalues of the Laplacian (used to compute diffusion);
- `gradX, gradY`: (NxN matrix) the gradient operator split into the x and y components of the tangent vectors.

If you don't know what's in a tensor, it's always useful to check the size. This is very helpful during debugging as well.

In [ ]:
print(f'frames: {frames.shape}')
print(f'mass: {mass.shape}')
print(f'L: {L.shape}')
print(f'evals: {evals.shape}')
print(f'evecs: {evecs.shape}')
print(f'gradX: {gradX.shape}')
print(f'gradY: {gradY.shape}')

### Diffusion

DiffusionNet combines two core operations: 1) Learning diffusion times using the Laplacian and 2) Computing directional features with gradients.

Let's first understand diffusion on our input shape. Diffusion can be understood as modeling how heat dissipates over a conducting surface over time. Let's say we add a source of heat at a single vertex. We can model that on our mesh by setting the value on one vertex to 1 and the rest to 0.

$\rightarrow$ Can you find the source? Hint: it's colored yellow and close to the sink.

In [ ]:
vertex_source = 3
heat_0 = torch.zeros(verts.shape[0])
heat_0[vertex_source] = 1
mp.plot(verts.numpy(), c=heat_0.numpy(), shading={"point_size": 0.1})

The _change_ of the heat distribution $x$ at time $t$ is given by the following differential equation
$$\frac{d}{dt} x(t) = \Delta x(t),$$
Where $\Delta$ is the Laplacian operator.

The distribution at time $t$ can be computed in a couple of ways. You may be familiar with the implicit computation using a backward Euler step:
$$(\mathbf{M} - t\mathbf{L})x_t = x_0,$$
where $\mathbf{M}$ is the mass matrix and $\mathbf{L}$ a discrete Laplacian operator (such as the Laplace-Beltrami operator on surfaces). We can compute diffusion by solving this linear system. You'll notice that it takes quite some time, depending on the size of the Laplacian. Try running with several values for t.

In [ ]:
import time
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve

# Set up the linear system Ax = b, where A = (M - tL) and b = heat_0
# First convert L and M to Scipy's sparse matrices
# Note that the sign convention for the Laplacian is different from the equation above.
L_sp = sp.csc_matrix(L.to_dense().numpy())
M = sp.diags(mass.numpy())

In [ ]:
# Compute heat diffusion implicitly
t = 1
A = M + t * L_sp

# Solve the system
start_time = time.perf_counter()
heat_t = spsolve(A, heat_0)
print(f'Time for solve: {time.perf_counter() - start_time}')

mp.plot(verts.numpy(), c=heat_t, shading={"point_size": 0.3})

Note two things here: if we change $t$, we change the left-hand-side of the equation. Since the goal is to learn diffusion times, we cannot make use of factorization. Luckily, we can accelerate this equation using an eigendecomposition of the Laplacian. You may be familiar with this kind of acceleration with convolutions on images in the Fourier domain (the eigendecomposition of the Laplacian on flat domains).

Diffusion for time $t$ is computed using the eigenbasis $U$ and eigenvalues $\lambda_0, \lambda_1, \ldots$ as follows:
$$x_t = U \begin{bmatrix}
e^{-\lambda_0 t} \\
e^{-\lambda_1 t} \\
\ldots
\end{bmatrix}\odot(U^TMx_0)$$

We can compute that with PyTorch as follows:

In [ ]:
t = 1
start_time = time.perf_counter()
power_coefs = torch.exp(-evals * t)
                                        # The mass matrix is a vector, so we don't implement it as matrix multiplication
heat_t_eig = evecs @ (power_coefs * (evecs.T @ (heat_0 * mass)))
print(f'Time for diffusion with eigendecomposition: {time.perf_counter() - start_time}')

mp.plot(verts.numpy(), c=heat_t_eig.numpy(), shading={"point_size": 0.3})

**Notice any difference?** You should! We computed diffusion time using the eigenvectors corresponding to only the 128 lowest eigenvalues of the Laplacian. That means the diffusion approximation can only represent **low** frequencies (high smoothing). If you want to compute more eigenvectors, that's quite expensive, especially for larger meshes. This is good to be aware of when you're scaling up or want to learn information on scales between the very high frequencies (computed with gradient features) and low frequencies.

$\rightarrow$ Try placing the source at different spots. Can you see how the diffusion is different? It turns out that the heat at the source point at several times is a pretty good descriptor of shapes. Learn more about that in the paper on the Heat Kernel Signature: https://www.lix.polytechnique.fr/~maks/papers/hks.pdf

$\rightarrow$ Try loading the same shape several times. Since we added a random rotation as transform, it should be rotated differently every time. Notice that the diffusion does not change!

### Gradients
A second part of DiffusionNet is learning directional information using the gradients of the features. Let's see what these gradients look like on the mesh. For simplicity, we'll use the heat distribution at time $t$ as the input.

In [ ]:
grad_x, grad_y = gradX @ heat_t_eig, gradY @ heat_t_eig
# Convert tangent-space coordinates to 3D using the frames
grad_3D = frames[:, 0] * grad_x.unsqueeze(-1) + frames[:, 1] * grad_y.unsqueeze(-1)

# We have to visualize in Polyscope, because meshplot doesn't support vector outputs
ps.init()

ps_point_cloud = ps.register_point_cloud("Input cloud", verts.numpy(), enabled=True)
ps_point_cloud.add_scalar_quantity("Heat distribution", heat_t_eig.numpy(), enabled=True)
ps_point_cloud.add_vector_quantity("Gradient of x", grad_3D.numpy(), enabled=True)

ps.show()

$\rightarrow$ What would happen to the gradient if the shape is rotated?

## Step 3: Setting up the model
Now that we have some understanding of the operations in DiffusionNet, we'll use the implementation provided by the authors to set up a model that we will train.

In [ ]:
# We'll use 3D features for this tutorial, but you can experiment with other features such as HKS as well.
C_in = 3
C_out = train_dataset.num_classes

# Create the model
model = diffusion_net.layers.DiffusionNet(
            C_in=C_in,
            C_out=C_out,
            C_width=32, # Internal size of the DiffusionNet blocks, we'll keep it small for now
            outputs_at='global_mean')

## Step 4: Train the model

Now that we have a model, we can start training. We do this with one of PyTorch's built in optimizers ([Adam](https://arxiv.org/pdf/1412.6980), a gradient descent optimizer using adaptive momentum).

$\rightarrow$ Follow the comments to understand the code.

**Note:** To keep the tutorial simple, we use a batch size of 1 and compute the operators during the training iteration. This is _not_ the recommended way to run DiffusionNet, as it is very slow! See correct examples of how to use DiffusionNet at https://github.com/nmwsharp/diffusion-net

In [ ]:
batch_size = 1

# Create data loaders for the training and test datasets
# We let the train data loaders automatically batch the data and shuffle it for training with the `shuffle=True` argument
train_loader = pyg.loader.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = pyg.loader.DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Create the model
model = model.to(device) # Move the model to the GPU if available
# Create the optimizer.
# We let the optimizer know which parameters to optimize and the learning rate (step size).
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
# We create the loss function, here a Cross Entropy Loss, typically used for classification tasks.
criterion = torch.nn.CrossEntropyLoss()

# Define the training loop
def train():
    # Set the model to training mode, which enables certain features like dropout, if they are used.
    model.train()

    # Initialize a total loss accumulator for reporting.
    total_loss = 0
    for data in tqdm(train_loader):
        # First set all the gradients to zero, because we accumulate gradients by default in PyTorch.
        optimizer.zero_grad()
        faces = torch.tensor([], device=data.pos.device)
        operators = get_operators(data.pos, torch.tensor([]))
        operators = [operator.to(device) for operator in operators[1:]] # Skip the frames
        data = data.to(device)  # Move the data to the GPU if available

        # Forward pass: compute the logits (predictions) for the input data.
        logits = model(data.pos, *operators).unsqueeze(0)
        # Compute the loss between the logits and the ground truth labels.
        loss = criterion(logits, data.y)

        # Backward pass: compute the gradients of the loss with respect to the model parameters.
        loss.backward()
        # Update the model parameters using the optimizer.
        optimizer.step()
        # Accumulate the loss for reporting.
        total_loss += float(loss) * data.num_graphs

    return total_loss / len(train_loader.dataset)

# We do not want to compute gradients during testing, so we use torch.no_grad()
# This saves memory and speeds up the computation.
@torch.no_grad()
def test():
    # Set the model to evaluation mode, which disables features like dropout.
    model.eval()

    total_correct = 0
    for data in tqdm(test_loader):
        faces = torch.tensor([], device=data.pos.device)
        operators = get_operators(data.pos, torch.tensor([]))
        operators = [operator.to(device) for operator in operators[1:]] # Skip the frames
        data = data.to(device)  # Move the data to the GPU if available
        logits = model(data.pos, *operators).unsqueeze(0)
        pred = logits.argmax(dim=-1)
        total_correct += int((pred == data.y).sum())

    return total_correct / len(test_loader.dataset)

# Training loop for 1 epoch
print('Training')
loss = train()
print('Testing')
test_acc = test()
print(f'Loss: {loss:.4f}, Test Acc: {test_acc:.4f}')
